# VN-RSI: Modular Micro-RSI Training Notebook (Free Kaggle `T4 x2` & Google Colab `T4`)

**Author**: Tran Duy Khang — AI Software Engineer / Freelance Vibe Researcher

This notebook trains the decomposed 3-model **VN-RSI** architecture on **FREE cloud GPUs ($0)**:
1. **Model 1 (`Qwen3-4B` or `Qwen3-1.7B`)**: 4-bit QLoRA via Unsloth (`~4.2 GB VRAM` on GPU 0)
2. **Model 2 (`DiscreteASTDiffusion`, ~6.4M params)**: Reward-Weighted ReST Masked Diffusion over causal DSL ASTs (`<0.2 GB VRAM` on GPU 1 / GPU 0)
3. **Model 3 (`HRMReasoner`, ~6.8M params)**: Hierarchical Slow ($z_H$) / Fast ($z_L$) Recursive Reasoner (`<0.2 GB VRAM` on GPU 1 / GPU 0)

> **Research Simulation ONLY**: Strictly no live trading or brokerage APIs.

In [ ]:
# 1. Check free GPU topology (Kaggle T4 x2 or Colab T4 16GB)
!nvidia-smi

In [ ]:
# 2. Clone your GitHub repository (update URL if you use a different repo name)
import os
REPO_URL = "https://github.com/KenyaWashed/VN_Trading_Agents.git"
if not os.path.exists("VN_Trading_Agents"):
    !git clone {REPO_URL}
%cd VN_Trading_Agents

In [ ]:
# 3. Install core VN-RSI package + Unsloth for 4-bit QLoRA on free T4 GPUs
!pip install -q -e ".[dev]"
!pip install -q unsloth trl datasets bitsandbytes accelerate

In [ ]:
# 4. Run mandatory anti-leakage & sandbox seal unit tests before training
!pytest tests/ -q

In [ ]:
# 5. Train Discrete AST Diffusion (~6.4M) + HRM Reasoner (~6.8M) + Qwen3-4B Unsloth 4-bit QLoRA
# On Colab 1x T4 (16GB), if you prefer 2x faster training, swap to --slm-model unsloth/Qwen3-1.7B-bnb-4bit
!python train_colab_kaggle.py \
    --preset kaggle_t4_optimal \
    --enable-unsloth-qlora \
    --slm-model unsloth/Qwen3-4B-bnb-4bit \
    --epochs 25 \
    --outer-iters 4

In [ ]:
# 6. Run full sealed evaluation & auto-verify LaTeX paper tables
!python experiments/run_data.py
!python experiments/run_baselines.py --reset-ledger
!python experiments/run_agent_v0.py
!python experiments/run_rsi.py
!python experiments/run_modular_rsi.py --preset kaggle_t4_optimal
!python experiments/build_report.py
!python paper/verify_paper.py